# Подготовка, обучение и разбор событий в одном ноутбуке

Выполняйте ячейки сверху вниз. Настройки, аудит режимов, реестр событий, обучение, тревоги и интерпретация выводятся здесь. Отдельно запускать `.py` и редактировать отчёты в CSV не требуется.

Реальные данные остаются у вас. Причины событий и назначения датчиков требуют проверки специалистом; отсутствие достаточной разметки выводится как отдельный статус.

Живые графики и требования классификаторов: [docs/ГРАФИКИ_И_ГОТОВНОСТЬ_РИСКА.md](docs/ГРАФИКИ_И_ГОТОВНОСТЬ_РИСКА.md).


In [ ]:
from pathlib import Path
import os, sys
import pandas as pd
from IPython.display import display

_candidates = [Path.cwd(), Path.cwd()/"training"]
if "__file__" in globals():
    _candidates.insert(0, Path(__file__).resolve().parent)
TRAINING_DIR = next((p for p in _candidates if (p/"lib/monitoring_v3").is_dir()), None)
if TRAINING_DIR is None:
    raise FileNotFoundError("Откройте ноутбук из папки проекта или training.")
os.chdir(TRAINING_DIR)
for _path in (TRAINING_DIR, TRAINING_DIR/"lib"):
    if str(_path) not in sys.path:
        sys.path.insert(0, str(_path))

from monitoring_v3.config import load
from monitoring_v3.data import load_sources
import monitoring_v3.pipeline as LF
from monitoring_v3.notebook_live import LiveTrainingReport
from notebook_reports import (
    configure, notebook_config, show_table, show_preparation, show_results, show_detail,
    episode_catalog, apply_episode_labels, run_interpretation, run_research,
)


## 1. Все настройки

Пути и группы берутся из `incremental_settings.json`, затем применяются `SETTINGS`. В этом ноутбуке явно выбран основной расчёт CatBoost / conditions / local: сравнения включаются отдельно и увеличивают время.

Изменённые данные/правила не продолжают несовместимые веса: нужные части пересчитываются с резервной копией в прежней output. После обновления качества необходим один пересчёт по новому контракту; далее новые месяцы продолжают модель. Подробности и все новые параметры — в [КАЧЕСТВО_И_ПЕРЕНОС.md](docs/КАЧЕСТВО_И_ПЕРЕНОС.md), полный справочник — [ИНСТРУКЦИЯ.md](ИНСТРУКЦИЯ.md).

Повторная проверка: [ошибки, исправления и влияние на результаты](docs/ПОВТОРНАЯ_ПРОВЕРКА.md). Текущий экспорт monitoring-3.2, контракт 4.


In [ ]:
SETTINGS = {
    "models": ["CatBoost"],
    "feature_sets": ["conditions"],
    "scenarios": ["local"],
    # "data_paths": {"К-5": "D:/private/К-5/hourly.parquet"},
    # "groups": {"БКЗ-500": ["К-5", "К-6", "К-7"]},
    # "first_test": "2025-03-01",
    # "run_thresholds": {"К-5": ...},  # физически проверенный порог
    # "startup_hours": 24,
    # "output": "runs/my_experiment",
    # "validate_candidate": True,
    # "history_policy": {"bootstrap": True, "reference_days": 60},
    # "regime_policy": {"stop_hours": 2, "current_pairs": {}},
    # "risk_policy": {"scope": "group", "samples": "episodes", "sample_hours": 6},
    # "critical_sensors": [],
    # "event_overrides": [],
}
SENSOR_RULES = {
    "defaults": {},
    "sensors": {
        # "общий параметр из аудита": {"kind": "rare", "mask_frozen": False},
    },
}
DETAIL = {
    "experiment": None,  # None — первый рассчитанный эксперимент; названия будут в таблице
    "unit": None,        # None — первый доступный котёл
    "view": "mean::instant",
    "style": "legacy",   # прежние 2 панели; diagnostic — подробный график
    "budget": None,      # None — основной бюджет из настроек
    "start": None,       # например "2025-03-01"
    "end": None,
    "importance_sensor": None, # общий ключ цели; None — первая обученная цель
    "importance_month": None,  # например "2025-05"; None — последняя версия
}
INTERPRETATION = {"run": True, "as_of": None, "attach": True}
# as_of=None — конец доступных данных; для группового риска самый поздний конец группы; полнота будущего каждого котла проверяется отдельно.
# attach=True — добавить обученные классификаторы в экспорт выбранных экспериментов.
# Только вывод: эти параметры не меняют веса/выборки основного детектора.
LIVE = {
    "enabled": True,                 # отдельный блок + прежний график на модель/котёл
    "update_every_months": 1,         # обновлять график после каждого N-го среза
    "risk_history": True,            # отдельное помесячное обучение риска для честных подписей
    "view": "mean::instant",         # этот же детектор запускает samples="episodes"
    "budget": None,                  # None — CFG["alarm_budget"]
    "start": None, "end": None,     # границы картинки; обучение не ограничивают
}
RESEARCH = {
    "ensemble": False, "survival": False, "sensitivity": False,
    "as_of": None, "sensor": None, "start": None,
}

CFG = notebook_config(SETTINGS, SENSOR_RULES)
show_table("Действующие настройки", pd.DataFrame(
    {"параметр": list(CFG), "значение": [str(value) for value in CFG.values()]}))


## 2. Данные, режимы и события до обучения

Ниже появятся фактические пороги нагрузки, граница выключения, часы режимов, качество каналов, токи и найденные пары приводов, а также полный реестр подтверждённых остановов и документов. Неизвестная причина не становится отрицательной меткой риска. Проверьте физический смысл пар МВ/ММТ и ПСУ по своей схеме.

Большие таблицы раскрываются нажатием на название. Ошибки источников исправляются в `SETTINGS`; затем повторите настройки и подготовку. JSON перечитывается перед подготовкой; график строится по текущим данным. При известных токах добавляется график состава работающих приводов и конфликтов.

После неизвестной нагрузки есть защитное восстановление (по умолчанию startup_hours); старый возраст не считается достоверным. В таблице токов показано число положительных опорных измерений.


In [ ]:
CFG = notebook_config(SETTINGS, SENSOR_RULES)
CONTEXTS, ISSUES = load_sources(CFG, Path(CFG["output"])/"audit", strict=False)
EVENTS = show_preparation(CONTEXTS, ISSUES, CFG)


## 3. Последовательное обучение и обновление

При `history_policy.bootstrap=True` причинные месячные модели начинают строиться по ранней истории после опорного периода подготовки; `first_test` задаёт начало основного отчёта. Это даёт классификатору более ранние признаки. Счётчик месяцев включает эти ранние срезы.

Первый fit требует достаточно обучающей нормы и независимой калибровки; пустая последняя неделя не мешает использовать старые рабочие кампании. При отсутствии новой нормы используется прежняя модель. Отдельного резерва проверки при `validate_candidate=False` нет. Часы калибровки никогда не переходят в последующий train.

В Jupyter обновляется один прогресс-бар с этапом, датчиком и временем; полный журнал остаётся в `<output>/progress.jsonl`. Готовая модель экспортируется до расчёта оценок месяца, пути записываются в `models.csv`. Готовые месячные состояния и оценки восстанавливаются при совместимости. Несовместимые состояния копируются в резерв и пересчитываются автоматически; вручную удалять fingerprint или искать новую output для исправления ошибки не требуется.

Продолжение CatBoost использует временный файл прежнего ансамбля, сохраняя деревья и добавляя новые. При отзыве уже обученной нормы выполняется явное пересоздание; при нехватке данных оценки становятся неизвестными.


Каждая модель/котёл имеет отдельный обновляемый блок и прежний двухпанельный график. Проценты месячных срезов и датчиков разделены. Риск дополняет эти же графики после раздела 7; при pooled нужно сначала получить признаки всей группы.


In [ ]:
if ISSUES:
    raise ValueError("Исправьте проблемы источников в SETTINGS и повторите аудит выше.")
LIVE_REPORT = LiveTrainingReport(CFG, LIVE)
RESULT = LF.run(CFG, live_report=LIVE_REPORT)


## 4. Результаты моделей и месячных обновлений

Показаны доступность и качество схем, сравнение одинаковых схем на общих часах, обучение по месяцам и точный отбор часов. Отдельные таблицы показывают ошибки целей в физических единицах, выигрыш перед константой, калибровку порогов и её ограниченность.

Первая полностью замороженная модель сравнивается с адаптацией на 1/3/6 месяцах и всём доступном периоде. Это позволяет проверить полезность обновлений, а не судить только по одному месяцу. Все файлы сохраняются автоматически.

Дополнительно показана сводная важность входов по обученным целям. Для каждой цели собственный список; важность не является причиной неисправности.

Часы до первой модели включены в доступность. working_unscored_hours отделяет работу без оценки от простоя; группа учитывает только собственные датчики.


In [ ]:
EXPERIMENTS = show_results(CFG, RESULT)
show_table("Эксперименты для выбора в DETAIL", EXPERIMENTS.drop(columns="directory", errors="ignore"), True)


## 5. График и события выбранного эксперимента

Выбор котла, модели, периода и схемы — в `DETAIL` в разделе настроек. График обрезается без пересчёта тревог. Ниже — обнаружение каждой аварии и эпизоды тревог с фактическим временем подтверждения.
Ниже — важность входов выбранной цели и версии. Выбор задаётся `DETAIL["importance_sensor"]` и `DETAIL["importance_month"]`; версия подписана отдельно от периода графика тревог.


In [ ]:
DETAIL_DIR = show_detail(CFG, DETAIL, CONTEXTS)


## 6. Проверка смысла эпизодов прямо в ноутбуке

Скопируйте идентификатор из каталога и добавьте запись в `EPISODE_LABEL_EDITS` ниже. Укажите эксперимент, котёл, смысл, специалиста и дату, когда эта метка стала известна. Это сохраняет вашу разметку автоматически; открывать CSV не требуется.

`suggestions` — гипотезы для проверки, а не подтверждённые причины. После изменения списка выполните ячейку сохранения меток и следующую ячейку интерпретации.

In [ ]:
CATALOG = episode_catalog(CFG)
show_table("Каталог эпизодов для разбора", CATALOG, True)


In [ ]:
EPISODE_LABEL_EDITS = [
    # {
    #     "experiment": "название из таблицы", "unit": "К-5",
    #     "episode_id": "идентификатор из каталога", "label": "mode_change",
    #     "label_available_at": "2025-04-01", "reviewer": "специалист",
    #     "reason": "подтверждено переключение оборудования",
    # },
]
if EPISODE_LABEL_EDITS:
    CATALOG = apply_episode_labels(CFG, EPISODE_LABEL_EDITS)
    show_table("Сохранённые метки", CATALOG.loc[CATALOG.label.notna()], True)
else:
    print("Новых проверенных меток нет; прежняя разметка сохранена.")


## 7. Интерпретация отклонений и предаварийного риска

Риск по умолчанию собирает события котлов группы. Одна строка — решение при подтверждённой тревоге и затем каждые 6 часов активного эпизода; на вход идут текущие признаки и окна 6/24/48 часов. Неизвестные остановы и неполное будущее не становятся отрицательным классом.

Достаточность рабочего обучения, калибровки вероятностей и исторической проверки показана отдельно. `trained_uncalibrated` означает: модель обучена, есть балл, но процента риска пока нет. `single_class_test` означает, что тест не подтверждает обнаружение аварий; он не запрещает рабочее обучение. Причины `cause` требуют проверенных меток эпизодов; без них остаются неизвестными.

Успешные доступные головы автоматически присоединяются к итоговому экспорту при `INTERPRETATION['attach']=True`. Итоговую голову нельзя применять задним числом раньше даты её доступности. `run=False` только показывает сохранённые статусы.

Для обученных голов автоматически выводятся таблицы и диаграммы важности их входов. Это глобальная опора модели, а не локальный диагноз эпизода.

При исторической проверке учитывается дата появления метки, а не только дата измерения. Рабочая голова сохраняется до дополнительных проверок. Публикация голов вместе с манифестом атомарна.


In [ ]:
INTERPRETATION_RESULT = run_interpretation(
    CFG, CONTEXTS, enabled=INTERPRETATION["run"],
    as_of=INTERPRETATION["as_of"], attach=INTERPRETATION["attach"],
    live_report=globals().get("LIVE_REPORT"), risk_history=LIVE["risk_history"],
)


## 8. Дополнительные исследования без отдельных команд

В `RESEARCH` можно включить сбор признаков нескольких детекторов (`ensemble`), исследование времени до отказа (`survival`) и проверку искусственными нарушениями (`sensitivity`). Для последней укажите `sensor` и `start`; используется модель из `DETAIL` и копия истории. Эти проверки не подтверждают качество на реальных авариях.

Вывод и таблицы появятся непосредственно под ячейкой.

In [ ]:
RESEARCH_RESULT = run_research(CFG, CONTEXTS, RESEARCH, DETAIL_DIR)
